# 🌍 Análise de Indicadores de Saúde - Região EMRO (Mediterrâneo Oriental)

Este notebook demonstra como utilizar o `epidemiological-datasets` para analisar dados de saúde da região **EMRO** (Eastern Mediterranean Regional Office) da OMS.

## 📋 Conteúdo
1. Introdução à região EMRO
2. Listagem de países da região
3. Análise de indicadores de saúde
4. Visualização de dados regionais
5. Comparação entre países

---

## 1️⃣ Configuração e Imports

In [ ]:
# Imports necessários
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from epidatasets.sources.who_ghoclient import WHOAccessor

# Configuração de visualização
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

%matplotlib inline

# Inicializar o acessor WHO
who = WHOAccessor()
print("✅ WHO Accessor inicializado com sucesso!")

## 2️⃣ Conhecendo a Região EMRO

A região EMRO (Eastern Mediterranean Regional Office) da OMS compreende **22 países** do Mediterrâneo Oriental e Oriente Médio.

In [ ]:
# Obter lista de países EMRO
emro_countries = who.get_emro_countries()
print(f"📊 Total de países na região EMRO: {len(emro_countries)}\n")

# Visualizar os países
emro_countries

### 📍 Distribuição Geográfica

Os países EMRO podem ser agrupados em sub-regiões:

In [ ]:
# Agrupar países por sub-região
sub_regions = {
    "Magreb & Norte da África": ["MAR", "DZA", "TUN", "LBY", "EGY", "SDN"],
    "Oriente Médio": ["LBN", "SYR", "JOR", "IRQ", "PSE", "ISR", "KWT", "BHR", "QAT", "ARE", "OMN", "SAU"],
    "Ásia Central & Sul": ["IRN", "PAK", "AFG"],
    "Chifre da África": ["SOM", "DJI"],
    "Península Arábica": ["YEM"]
}

for region, codes in sub_regions.items():
    countries_in_region = emro_countries[emro_countries['country_code'].isin(codes)]['country_name'].tolist()
    if countries_in_region:
        print(f"\n🌐 {region}:")
        for country in countries_in_region:
            print(f"   • {country}")

## 3️⃣ Análise de Indicadores de Saúde

### 3.1 Expectativa de Vida Saudável (HALE)

Vamos analisar a **Healthy Life Expectancy (HALE)** ao nascer para toda a região EMRO.

In [ ]:
# Buscar dados de HALE para EMRO
hale_data = who.get_emro_life_expectancy(years=list(range(2015, 2022)))

print(f"📈 Registros obtidos: {len(hale_data)}")
print(f"\nColunas disponíveis: {list(hale_data.columns)}")

# Visualizar primeiras linhas
hale_data.head(10)

### 📊 Visualização da Expectativa de Vida por País

In [ ]:
# Filtrar dados mais recentes
latest_year = hale_data['year'].max()
latest_hale = hale_data[hale_data['year'] == latest_year].copy()

# Ordenar por valor
latest_hale = latest_hale.sort_values('value', ascending=True)

# Criar gráfico de barras
fig, ax = plt.subplots(figsize=(12, 8))

bars = ax.barh(latest_hale['country_code'], latest_hale['value'], 
               color=sns.color_palette("viridis", len(latest_hale)))

ax.set_xlabel('Expectativa de Vida Saudável (anos)', fontsize=12)
ax.set_ylabel('País (código ISO3)', fontsize=12)
ax.set_title(f'Expectativa de Vida Saudável (HALE) - Região EMRO ({latest_year})', 
             fontsize=14, fontweight='bold')

# Adicionar valores nas barras
for i, (bar, value) in enumerate(zip(bars, latest_hale['value'])):
    ax.text(value + 0.5, bar.get_y() + bar.get_height()/2, 
            f'{value:.1f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\n📊 Estatísticas da região EMRO ({latest_year}):")
print(f"   Média: {latest_hale['value'].mean():.1f} anos")
print(f"   Mediana: {latest_hale['value'].median():.1f} anos")
print(f"   Mínimo: {latest_hale['value'].min():.1f} anos ({latest_hale.iloc[0]['country_code']})")
print(f"   Máximo: {latest_hale['value'].max():.1f} anos ({latest_hale.iloc[-1]['country_code']})")

### 3.2 Tendências Temporais da Região

In [ ]:
# Obter tendências regionais agregadas
hale_trends = who.get_emro_health_trends(
    indicator="WHOSIS_000002",  # HALE
    start_year=2015,
    end_year=2022
)

print("📈 Tendências regionais de HALE:")
hale_trends

In [ ]:
# Visualizar tendências
fig, ax = plt.subplots(figsize=(12, 6))

ax.fill_between(hale_trends['year'], 
                hale_trends['emro_mean'] - hale_trends['emro_std'],
                hale_trends['emro_mean'] + hale_trends['emro_std'],
                alpha=0.3, label='±1 Desvio Padrão')

ax.plot(hale_trends['year'], hale_trends['emro_mean'], 
        marker='o', linewidth=2, markersize=8, label='Média EMRO')

ax.plot(hale_trends['year'], hale_trends['emro_median'], 
        marker='s', linewidth=2, markersize=8, label='Mediana EMRO', linestyle='--')

ax.set_xlabel('Ano', fontsize=12)
ax.set_ylabel('Expectativa de Vida Saudável (anos)', fontsize=12)
ax.set_title('Tendências de HALE na Região EMRO (2015-2022)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4️⃣ Análise de Malária na Região

A malária é um problema significativo de saúde pública em partes da região EMRO, especialmente no Afeganistão, Paquistão, Somália, Sudão e Iêmen.

In [ ]:
# Buscar dados de malária
malaria_data = who.get_emro_malaria_data(years=[2018, 2019, 2020, 2021, 2022])

print(f"🦟 Registros de malária obtidos: {len(malaria_data)}")

# Verificar dados disponíveis
if not malaria_data.empty:
    print(f"\nPaíses com dados: {malaria_data['country_code'].nunique()}")
    print(f"Anos disponíveis: {sorted(malaria_data['year'].unique())}")
    malaria_data.head(10)
else:
    print("⚠️ Nenhum dado de malária disponível para os anos selecionados")

In [ ]:
# Análise de incidência de malária (se dados disponíveis)
if not malaria_data.empty and 'value' in malaria_data.columns:
    # Agrupar por país e calcular média
    malaria_by_country = malaria_data.groupby('country_code')['value'].mean().sort_values(ascending=False)
    
    # Mostrar países com maior incidência
    print("🦟 Países com maior incidência média de malária:")
    print(malaria_by_country.head(10))
    
    # Visualizar
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Filtrar apenas países com dados significativos
    significant = malaria_by_country[malaria_by_country > 0].head(15)
    
    if len(significant) > 0:
        bars = ax.barh(significant.index[::-1], significant.values[::-1], 
                       color='coral')
        
        ax.set_xlabel('Incidência Estimada de Malária', fontsize=12)
        ax.set_ylabel('País', fontsize=12)
        ax.set_title('Incidência Média de Malária - Região EMRO', 
                     fontsize=14, fontweight='bold')
        
        plt.tight_layout()
        plt.show()

## 5️⃣ Comparação entre Países

Vamos comparar múltiplos indicadores entre os países da região.

In [ ]:
# Comparar HALE entre países EMRO
hale_comparison = who.compare_emro_countries(
    indicator="WHOSIS_000002",
    years=[2020, 2021]
)

print("📊 Comparação de HALE entre países EMRO:")
hale_comparison.head(10)

In [ ]:
# Heatmap de comparação
if not hale_comparison.empty:
    # Selecionar top 10 países mais recentes
    recent_data = hale_comparison.loc[2021] if 2021 in hale_comparison.index else hale_comparison.iloc[-1]
    top_countries = recent_data.dropna().sort_values(ascending=False).head(10).index
    
    # Criar heatmap
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Transpor para visualização
    heatmap_data = hale_comparison[top_countries].T
    
    sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='RdYlGn', 
                linewidths=0.5, ax=ax, cbar_kws={'label': 'HALE (anos)'})
    
    ax.set_title('Expectativa de Vida Saudável - Comparação entre Países EMRO', 
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Ano', fontsize=12)
    ax.set_ylabel('País', fontsize=12)
    
    plt.tight_layout()
    plt.show()

## 6️⃣ Análise Customizada por Indicador

Você pode buscar qualquer indicador da OMS para a região EMRO.

In [ ]:
# Exemplo: Taxa de mortalidade infantil
try:
    under5_mortality = who.get_emro_indicator(
        indicator="MDG_0000000001",  # Under-five mortality rate
        years=[2019, 2020, 2021, 2022]
    )
    
    print(f"👶 Registros de mortalidade infantil: {len(under5_mortality)}")
    
    if not under5_mortality.empty:
        # Visualizar
        latest = under5_mortality[under5_mortality['year'] == under5_mortality['year'].max()]
        latest_sorted = latest.sort_values('value', ascending=True)
        
        fig, ax = plt.subplots(figsize=(12, 8))
        
        bars = ax.barh(latest_sorted['country_code'], latest_sorted['value'],
                       color=sns.color_palette("Reds", len(latest_sorted)))
        
        ax.set_xlabel('Taxa de Mortalidade Infantil (por 1,000 nascidos vivos)', fontsize=12)
        ax.set_title('Mortalidade Infantil (Under-5) - Região EMRO', 
                     fontsize=14, fontweight='bold')
        
        plt.tight_layout()
        plt.show()
        
except Exception as e:
    print(f"⚠️ Erro ao buscar indicador: {e}")
    print("💡 Tente outro código de indicador disponível na OMS")

## 7️⃣ Explorando Outros Indicadores

Você pode pesquisar outros indicadores disponíveis na OMS.

In [ ]:
# Pesquisar indicadores relacionados a doenças
disease_indicators = who.search_indicators("dengue")
print(f"🔍 Indicadores relacionados a dengue: {len(disease_indicators)}")
disease_indicators.head(10)

---

## 📚 Resumo e Conclusões

Este notebook demonstrou como utilizar o `epidemiological-datasets` para:

1. **Listar países** da região EMRO (22 países)
2. **Buscar indicadores** específicos para toda a região
3. **Analisar tendências** temporais agregadas
4. **Comparar países** entre si
5. **Visualizar dados** de forma clara e informativa

### 🚀 Próximos Passos

- Explore outros indicadores da OMS usando `who.search_indicators()`
- Combine dados de múltiplas fontes (ex: dados climáticos + doenças)
- Crie modelos preditivos para a região
- Exporte os dados para análise em outras ferramentas

### 📖 Documentação

- WHO GHO API: https://www.who.int/data/gho
- EMRO Countries: https://www.emro.who.int/countries.html
- epidemiological-datasets: Documentação do pacote

In [ ]:
# Exportar dados para CSV (opcional)
# hale_data.to_csv('emro_hale_data.csv', index=False)
# print("✅ Dados exportados para 'emro_hale_data.csv'")